# Q1. Communication-Avoiding TSQR Algorithm implementation

In [4]:
import numpy as np

def tsqr(W, num_blocks=4):
    # Getting size of m by b matrix
    m, b = W.shape

    # Splitting W into 4 row blocks
    block_size = m // num_blocks
    blocks = []
    for i in range(num_blocks):
        start = i * block_size
        end = (i+1) * block_size
        blocks.append(W[start:end, :])

    # Stage 1: Local QR on each block  
    # Each processor factorises its own chunk independently
    Q_local = []
    R_local = []
    for i in range(num_blocks):
        Qi, Ri = np.linalg.qr(blocks[i])
        Q_local.append(Qi)
        R_local.append(Ri)

    # Stage 2: Reduction of pairs of R's
    # Stack R factors in pairs and QR factorise the small (2b x b) matrices
    R_stacked_01 = np.vstack([R_local[0], R_local[1]])
    Q_01, R_01 = np.linalg.qr(R_stacked_01)

    R_stacked_23 = np.vstack([R_local[2], R_local[3]])
    Q_11, R_11 = np.linalg.qr(R_stacked_23)

    # Stage 3: Final reduction
    # Stack the two remaining R factors and factorise
    R_stacked_final = np.vstack([R_01, R_11])
    Q_02, R_full = np.linalg.qr(R_stacked_final)

    # Reconstructing Q
    # Splitting Q_02 into top and bottom halves
    Q_02_top = Q_02[:b, :]
    Q_02_bot = Q_02[b:, :]

    # Propagating through stage 2
    # Q_01 @ Q_02_top gives a (2b x b) matrix, split into two (b x b) pieces
    mid_top = Q_01 @ Q_02_top 
    mid_bot = Q_11 @ Q_02_bot

    # Computing multipliers for each block
    mid_0 = mid_top[:b, :]
    mid_1 = mid_top[b:, :]
    mid_2 = mid_bot[:b, :]
    mid_3 = mid_bot[b:, :]

    # Propagating through stage 1: each local Q multiplies its piece
    Q_block_0 = Q_local[0] @ mid_0
    Q_block_1 = Q_local[1] @ mid_1
    Q_block_2 = Q_local[2] @ mid_2
    Q_block_3 = Q_local[3] @ mid_3

    Q_full = np.vstack([Q_block_0, Q_block_1, Q_block_2, Q_block_3])

    return Q_full, R_full



np.random.seed(42)
m, b = 200, 5
W = np.random.randn(m, b)

Q, R = tsqr(W)

# Comparing against numpy's QR
Q_ref, R_ref = np.linalg.qr(W)

print(f"Size of matrix W to solve: {W.shape}")
print()

# Checking if W = Q @ R
error = np.linalg.norm(W - Q @ R)
print(f"||W - QR||  = {error:.2e}")

# Checking if Q is orthogonal
orthogonality_error = np.linalg.norm(Q.T @ Q - np.eye(b))
print()
print("Checking if Q's are orthogonal:")
print(f"||Q'Q - I|| = {orthogonality_error:.2e}")

Size of matrix W to solve: (200, 5)

||W - QR||  = 9.06e-15

Checking if Q's are orthogonal:
||Q'Q - I|| = 1.19e-15
